In [1]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict
from tqdm import trange, tqdm
from sklearn.metrics import pairwise_distances

import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
path = '../MEC_04_002/OPTIR/'
dirs = os.listdir(path)
#del dirs[1]
del dirs[2]
dirs

['FB_L_90N_02_20240530', 'FB_L_120N_01_20240530', 'FB_L_60N_01_20240529']

In [51]:
files_list = []
for d in dirs:
    d_2 = os.listdir(path + d)
    files_list = files_list + [d+'/'+f for f in d_2 if f[-4:] == '.csv']#[d+'/'+f for f in d_2 if f[:6] == 'Mosaic']
#files_list
files_list

['FB_L_90N_02_20240530/FB_L_AlSi_90N_01_Mosaic_9_Image_9.csv',
 'FB_L_90N_02_20240530/FB_L_AlSi_90N_01_Mosaic_2_Image_3.csv',
 'FB_L_90N_02_20240530/FB_L_AlSi_90N_01_Mosaic_12_Image_3.csv',
 'FB_L_90N_02_20240530/FB_L_AlSi_90N_01_Mosaic_12_Image_2.csv',
 'FB_L_90N_02_20240530/FB_L_AlSi_90N_01_Mosaic_2_Image_2.csv',
 'FB_L_90N_02_20240530/FB_L_AlSi_90N_01_Mosaic_9_Image_8.csv',
 'FB_L_90N_02_20240530/FB_L_AlSi_90N_01_Mosaic_12_Image_1.csv',
 'FB_L_90N_02_20240530/FB_L_AlSi_90N_01_Mosaic_2_Image_1.csv',
 'FB_L_90N_02_20240530/FB_L_AlSi_90N_01_Mosaic_5_Image_9.csv',
 'FB_L_90N_02_20240530/FB_L_AlSi_90N_01_Mosaic_12_Image_5.csv',
 'FB_L_90N_02_20240530/FB_L_AlSi_90N_01_Mosaic_2_Image_5.csv',
 'FB_L_90N_02_20240530/FB_L_AlSi_90N_01_Mosaic_2_Image_4.csv',
 'FB_L_90N_02_20240530/FB_L_AlSi_90N_01_Mosaic_12_Image_4.csv',
 'FB_L_90N_02_20240530/FB_L_AlSi_90N_01_Mosaic_5_Image_8.csv',
 'FB_L_90N_02_20240530/FB_L_AlSi_90N_01_Mosaic_12_Image_6.csv',
 'FB_L_90N_02_20240530/FB_L_AlSi_90N_01_Mosaic_2_

In [59]:
files_codes = pd.merge(pd.DataFrame({'file_name':files_list,
                                     'key':[str(x[:2]) for x in files_list],
                                     'force':[str(x[5:7]) if x[7] == "N" else 
                                            str(x[5:8]) for x in files_list],
                                     'image':[str(x[-11:-4]) for x in files_list],
                                     'mosaic':[str(x[-20:-12]) if x[-14] == "_" else 
                                               str(x[-21:-12]) for x in files_list],
                                     #'material':[str(x[3:4]) if str(x[:2]) != "Bl" 
                                                 #else "-" for x in files_list]
                                     }),
                       pd.DataFrame({'key':np.unique([str(x[:2]) for x in files_list]),
                                     'brand':['FB']}))
files_codes = files_codes.set_index('file_name')
files_codes#.query('mosaic == "11"')

,key,force,image,mosaic,brand
file_name,,,,,
FB_L_90N_02_20240530/FB_L_AlSi_90N_01_Mosaic_9_Image_9.csv,FB,90,Image_9,Mosaic_9,FB
FB_L_90N_02_20240530/FB_L_AlSi_90N_01_Mosaic_2_Image_3.csv,FB,90,Image_3,Mosaic_2,FB
FB_L_90N_02_20240530/FB_L_AlSi_90N_01_Mosaic_12_Image_3.csv,FB,90,Image_3,Mosaic_12,FB
FB_L_90N_02_20240530/FB_L_AlSi_90N_01_Mosaic_12_Image_2.csv,FB,90,Image_2,Mosaic_12,FB
FB_L_90N_02_20240530/FB_L_AlSi_90N_01_Mosaic_2_Image_2.csv,FB,90,Image_2,Mosaic_2,FB
...,...,...,...,...,...
FB_L_60N_01_20240529/FB_L_AlSi_60N_01_Mosaic_5_Image_4.csv,FB,60,Image_4,Mosaic_5,FB
FB_L_60N_01_20240529/FB_L_AlSi_60N_01_Mosaic_5_Image_6.csv,FB,60,Image_6,Mosaic_5,FB
FB_L_60N_01_20240529/FB_L_AlSi_60N_01_Mosaic_11_Image_5.csv,FB,60,Image_5,Mosaic_11,FB


In [78]:
column_list = []
df_list = []
prev_mosaic = ''
prev_image = ''
count = 1
for f in files_list:
    df = pd.read_csv(path + f)
    column_list.append(df.columns)
    df.columns = ['feat_id', 'map_x', 'map_y', 'size', 'area', 'perimeter', 'circularity']
    mosaic = files_codes.loc[f, 'mosaic']
    image = files_codes.loc[f, 'image']
    if mosaic == prev_mosaic and image == prev_image:
        count = count + 1
    df['id'] = [files_codes.loc[f, 'brand'] + '_' + 
                files_codes.loc[f, 'force'] + '_' + 
                files_codes.loc[f, 'image'] + '_' + 
                files_codes.loc[f, 'mosaic'] + '_' +
                str(count) + '_' +
                str(i + 1) for i in range(df.shape[0])]
    
    #df['id_file'] = [files_codes.loc[f, 'brand'] + '_' + 
    #            files_codes.loc[f, 'material'] + '_' + 
    #            files_codes.loc[f, 'run'] for i in range(df.shape[0])]
    
    #df['brand'] = files_codes.loc[f, 'brand']
    #df['run'] = files_codes.loc[f, 'run']
    #df['mosaic'] = files_codes.loc[f, 'mosaic']
    df_list.append(df.iloc[:,1:])
    df_list.append(df)
    #prev_mosaic = mosaic
    
coord_df = pd.concat(df_list,axis=0)

In [82]:
coord_df.to_csv('MEC_04_002_mosaic_info.csv')